[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/11_agents_tools_mcp/40_designing_robust_tools.ipynb)

# 📓 Notebook 40 — Designing Robust Tools

> **Module:** Agents, Tools & MCP · **Estimated time:** 70–90 min · **Difficulty:** Intermediate → Advanced

An agent is only as good as its tools. Notebook 39 gave tools to a loop; here we make the **tools themselves** production-grade. A real tool is called by a model that may pass wrong types, missing fields, or hostile input — and may call it dozens of times. So a tool needs a **typed schema**, **input validation**, **structured errors**, **size limits**, an **approval gate** for dangerous actions, and the ability to run **in parallel**.

Everything runs offline. The patterns map 1:1 onto OpenAI/Anthropic function-calling and onto MCP tools (Notebook 41).

## 🎯 Learning objectives

By the end you can:

1. Describe a tool with a **JSON-Schema** the model can read.
2. **Validate** arguments before executing — types, required fields, enums, ranges.
3. Return **structured results and errors** (never raw exceptions) in a consistent envelope.
4. Build a **ToolRegistry** that validates, dispatches, and logs every call.
5. Add a **human-in-the-loop approval gate** for sensitive tools.
6. Run independent tools **in parallel** and bound output size.

## ✅ Prerequisites

Notebook 39 (agent architectures), dictionaries & functions (NB 4–5), JSON schemas (NB 4).

## 1. Anatomy of a tool

A tool the model can use has four parts:

| Part | Why the model needs it |
|---|---|
| **name** | what to call |
| **description** | *when* to call it — the single most important field |
| **parameters** (JSON-Schema) | the shape of the arguments |
| **function** | your code that actually runs (the model never runs it) |

We bundle them in a small dataclass.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Any
import json, time

@dataclass
class Tool:
    name: str
    description: str
    parameters: dict          # JSON-Schema for the arguments object
    fn: Callable[..., Any]

    def schema(self) -> dict:
        """The provider-facing description (OpenAI/Anthropic/MCP all use this shape)."""
        return {"name": self.name, "description": self.description,
                "parameters": self.parameters}

# Example tool: refund lookup over a tiny in-memory orders table
ORDERS = {"A-1": 19.0, "A-2": 49.0, "A-3": 5.0}

refund = Tool(
    name="estimate_refund",
    description="Estimate the refund owed for an order id. Use when a customer asks about money back.",
    parameters={
        "type": "object",
        "properties": {
            "order_id": {"type": "string", "description": "e.g. 'A-2'"},
            "pct":      {"type": "number", "minimum": 0, "maximum": 100,
                         "description": "percent to refund"},
        },
        "required": ["order_id", "pct"],
    },
    fn=lambda order_id, pct: {"order_id": order_id,
                              "refund": round(ORDERS[order_id] * pct / 100, 2)},
)
print(json.dumps(refund.schema(), indent=2))

## 2. Validate arguments *before* running

A model will, eventually, send `pct: "fifty"` or omit `order_id`. Running the function on bad input gives a confusing traceback. Instead, validate against the schema first and return a clear error the model can recover from.

We write a tiny validator (no extra dependency) covering the cases that actually bite: missing required fields, wrong types, enums, and numeric ranges.

In [ ]:
_JSON_TYPES = {"string": str, "number": (int, float), "integer": int,
               "boolean": bool, "object": dict, "array": list}

def validate_args(schema: dict, args: dict) -> list[str]:
    """Return a list of human-readable problems ([] means valid)."""
    problems = []
    props = schema.get("properties", {})
    for req in schema.get("required", []):
        if req not in args:
            problems.append(f"missing required field '{req}'")
    for key, val in args.items():
        if key not in props:
            problems.append(f"unexpected field '{key}'"); continue
        spec = props[key]
        exp = spec.get("type")
        if exp and not isinstance(val, _JSON_TYPES.get(exp, object)):
            problems.append(f"'{key}' should be {exp}, got {type(val).__name__}"); continue
        if "enum" in spec and val not in spec["enum"]:
            problems.append(f"'{key}' must be one of {spec['enum']}")
        if isinstance(val, (int, float)):
            if "minimum" in spec and val < spec["minimum"]:
                problems.append(f"'{key}' below minimum {spec['minimum']}")
            if "maximum" in spec and val > spec["maximum"]:
                problems.append(f"'{key}' above maximum {spec['maximum']}")
    return problems

print("valid  :", validate_args(refund.parameters, {"order_id": "A-2", "pct": 50}))
print("bad    :", validate_args(refund.parameters, {"order_id": "A-2", "pct": 150}))
print("missing:", validate_args(refund.parameters, {"pct": 50}))
print("type   :", validate_args(refund.parameters, {"order_id": "A-2", "pct": "fifty"}))

## 3. A consistent result envelope

Return the *same shape* whether a tool succeeds or fails. Models (and your logs) handle one predictable structure far better than a mix of values and exceptions:

```json
{"ok": true,  "result": {...}}
{"ok": false, "error": "estimate_refund: 'pct' above maximum 100"}
```

In [ ]:
def ok(result):  return {"ok": True,  "result": result}
def err(message): return {"ok": False, "error": message}

def safe_call(tool: Tool, args: dict) -> dict:
    problems = validate_args(tool.parameters, args)
    if problems:
        return err(f"{tool.name}: " + "; ".join(problems))
    try:
        return ok(tool.fn(**args))
    except Exception as e:                      # never leak a raw traceback to the model
        return err(f"{tool.name} raised {type(e).__name__}: {e}")

print(safe_call(refund, {"order_id": "A-2", "pct": 50}))
print(safe_call(refund, {"order_id": "A-2", "pct": 150}))
print(safe_call(refund, {"order_id": "NOPE", "pct": 50}))   # KeyError -> structured error

## 4. A ToolRegistry: validate, dispatch, log

In a real agent you have *many* tools. A registry gives you one place to register them, hand their schemas to the model, dispatch calls by name, and **log every invocation** for debugging and cost tracking.

In [ ]:
class ToolRegistry:
    def __init__(self):
        self._tools: dict[str, Tool] = {}
        self.log: list[dict] = []

    def register(self, tool: Tool):
        self._tools[tool.name] = tool
        return self

    def schemas(self) -> list[dict]:
        return [t.schema() for t in self._tools.values()]

    def call(self, name: str, args: dict) -> dict:
        t0 = time.perf_counter()
        if name not in self._tools:
            out = err(f"no such tool '{name}' (have: {list(self._tools)})")
        else:
            out = safe_call(self._tools[name], args)
        self.log.append({"tool": name, "args": args, "ok": out["ok"],
                         "ms": round((time.perf_counter() - t0) * 1000, 2)})
        return out

reg = ToolRegistry().register(refund)
reg.register(Tool("list_orders", "List all order ids and amounts.",
                  {"type": "object", "properties": {}}, lambda: dict(ORDERS)))

print(reg.call("estimate_refund", {"order_id": "A-1", "pct": 100}))
print(reg.call("list_orders", {}))
print(reg.call("delete_db", {}))            # unknown tool -> structured error
print("\ncall log:", json.dumps(reg.log, indent=1))

## 5. Tool selection among many

With one or two tools the model picks easily. With twenty, *descriptions* do the work — the model matches the user's intent to the best `description`. Offline, we mimic that with a keyword router; the lesson is the same: **a tool is only discoverable if its description says when to use it.**

In [ ]:
def route(user_msg: str, registry: ToolRegistry) -> str | None:
    """Stand-in for the model choosing a tool by reading descriptions."""
    msg = user_msg.lower()
    best, best_score = None, 0
    for schema in registry.schemas():
        words = set(schema["description"].lower().replace(".", " ").split())
        score = len(words & set(msg.split()))
        if score > best_score:
            best, best_score = schema["name"], score
    return best

print("‘how much money back for A-2?’ ->", route("how much money back for A-2", reg))
print("‘show me all orders’          ->", route("show me all the orders please", reg))

## 6. Approval gate: human-in-the-loop for dangerous tools

Reading data is safe; **issuing a refund, sending an email, or deleting a row is not.** Mark sensitive tools and require explicit approval before they execute. The gate is a callback — in production it's a Slack button or a UI prompt; here it's a simple policy function.

In [ ]:
SENSITIVE = {"estimate_refund"}     # tools that change money/state

def gated_call(registry, name, args, approver):
    if name in SENSITIVE:
        if not approver(name, args):
            return err(f"'{name}' denied by approver")
    return registry.call(name, args)

def auto_approver(name, args):
    # Example policy: auto-approve small refunds, escalate large ones.
    decision = not (name == "estimate_refund" and args.get("pct", 0) > 50)
    print(f"   approver: {name}({args}) -> {'APPROVE' if decision else 'DENY'}")
    return decision

print(gated_call(reg, "estimate_refund", {"order_id": "A-1", "pct": 25}, auto_approver))
print(gated_call(reg, "estimate_refund", {"order_id": "A-1", "pct": 90}, auto_approver))

## 7. Parallel tool calls

When a model requests several *independent* tools (e.g. look up three orders at once), run them concurrently instead of in series. Bound the pool so you never launch unbounded work.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def call_many(registry, calls: list[tuple[str, dict]], max_workers: int = 4) -> list[dict]:
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = [pool.submit(registry.call, name, args) for name, args in calls]
        return [f.result() for f in futures]

batch = [("estimate_refund", {"order_id": oid, "pct": 100}) for oid in ORDERS]
for r in call_many(reg, batch):
    print(r)

## 8. Bound the output

A tool that returns a 50,000-row table will blow your context window and your bill. Cap tool output — truncate, paginate, or summarise — and tell the model you did.

In [ ]:
def bounded(result, max_items: int = 3) -> dict:
    if isinstance(result, dict) and len(result) > max_items:
        keep = dict(list(result.items())[:max_items])
        return {"truncated": True, "shown": max_items, "total": len(result), "items": keep}
    return {"truncated": False, "items": result}

big = {f"row-{i}": i for i in range(100)}
print(bounded(big))

## 🧪 Practice exercises

### Exercise 1 — ⭐ Add an `enum` field

Add a `currency` argument to a tool, restricted to `["EUR", "USD"]`, and show `validate_args` rejects `"GBP"`.

In [ ]:
pay_schema = {"type": "object",
              "properties": {"currency": {"type": "string", "enum": ["EUR", "USD"]}},
              "required": ["currency"]}
print("USD:", validate_args(pay_schema, {"currency": "USD"}))
print("GBP:", validate_args(pay_schema, {"currency": "GBP"}))

### Exercise 2 — ⭐⭐ Count failures in the log

Write `failure_rate(registry)` returning the fraction of logged calls where `ok` is False.

In [ ]:
def failure_rate(registry) -> float:
    if not registry.log: return 0.0
    fails = sum(1 for e in registry.log if not e["ok"])
    return round(fails / len(registry.log), 3)

print("failure rate so far:", failure_rate(reg))

### Exercise 3 — ⭐⭐ A read-only registry view

Return just the *names and descriptions* of registered tools — what you'd show a user as "what can this agent do?".

In [ ]:
def capabilities(registry) -> list[str]:
    return [f"{s['name']}: {s['description']}" for s in registry.schemas()]
print("\n".join(capabilities(reg)))

### Exercise 4 — ⭐⭐ Debug me 🐞

This validator call is supposed to flag a too-large `pct`, but it reports *valid*. Find why (next cell fixes it).

In [ ]:
# 🐞 BUG (INTENTIONALLY ERRORS): args passed as a JSON *string*, not a dict.
bad = '{"order_id": "A-2", "pct": 999}'
problems = validate_args(refund.parameters, bad)   # AttributeError: str has no .items / wrong behaviour
print("problems:", problems)
assert problems, "expected the out-of-range pct to be caught"

In [ ]:
# ✅ Fix: parse the JSON to a dict first — tools receive dicts, not strings.
bad = json.loads('{"order_id": "A-2", "pct": 999}')
print("problems:", validate_args(refund.parameters, bad))

## 🧠 Stretch exercises

### Stretch A — ⭐⭐⭐ Idempotency keys

Make `estimate_refund` idempotent: a repeated call with the same `idempotency_key` returns the cached result instead of recomputing (so a retried request never double-refunds).

In [ ]:
_seen: dict[str, dict] = {}
def idempotent_refund(order_id, pct, idempotency_key):
    if idempotency_key in _seen:
        return {**_seen[idempotency_key], "replayed": True}
    res = {"order_id": order_id, "refund": round(ORDERS[order_id] * pct / 100, 2)}
    _seen[idempotency_key] = res
    return res

print(idempotent_refund("A-2", 50, "req-1"))
print(idempotent_refund("A-2", 50, "req-1"))   # replayed, not recomputed

### Stretch B — ⭐⭐⭐ Retry with backoff

Wrap a flaky tool so it retries on failure with exponential backoff, up to N attempts.

In [ ]:
def with_retry(fn, attempts=3, base=0.001):
    def wrapped(*a, **k):
        for i in range(attempts):
            out = fn(*a, **k)
            if out.get("ok", True):
                return out
            time.sleep(base * (2 ** i))
        return out
    return wrapped

calls = {"n": 0}
def flaky():
    calls["n"] += 1
    return ok("done") if calls["n"] >= 2 else err("transient")

print(with_retry(flaky)())          # succeeds on the 2nd try
print("attempts made:", calls["n"])

### Stretch C — ⭐⭐⭐ A timeout wrapper

Kill a tool call that runs longer than `seconds` and return a structured timeout error.

In [ ]:
def call_with_timeout(fn, seconds=0.05, *args, **kwargs):
    with ThreadPoolExecutor(max_workers=1) as pool:
        fut = pool.submit(fn, *args, **kwargs)
        try:
            return ok(fut.result(timeout=seconds))
        except Exception:
            return err(f"timeout after {seconds}s")

print(call_with_timeout(lambda: sum(range(1000)), 0.05))
print(call_with_timeout(lambda: time.sleep(1), 0.02))

### Stretch D — ⭐⭐⭐ Schema-driven argument coercion

Models sometimes send `"50"` where a number is wanted. Coerce string→number when the schema says `number`/`integer`, *then* validate.

In [ ]:
def coerce(schema, args):
    out = dict(args)
    for k, spec in schema.get("properties", {}).items():
        if k in out and spec.get("type") in ("number", "integer") and isinstance(out[k], str):
            try: out[k] = float(out[k]) if spec["type"] == "number" else int(out[k])
            except ValueError: pass
    return out

raw = {"order_id": "A-2", "pct": "50"}
fixed = coerce(refund.parameters, raw)
print("coerced:", fixed, "| valid:", validate_args(refund.parameters, fixed))

## 🎁 Bonus mini-project — a hardened tool call

Combine everything into one `production_call(registry, name, args, approver)` that **coerces → validates → gates → dispatches → logs**, and returns the standard envelope. This is the function an agent loop should actually call.

In [ ]:
def production_call(registry, name, args, approver=lambda n, a: True):
    if name not in registry._tools:
        return err(f"no such tool '{name}'")
    schema = registry._tools[name].parameters
    args = coerce(schema, args)
    problems = validate_args(schema, args)
    if problems:
        return err(f"{name}: " + "; ".join(problems))
    if name in SENSITIVE and not approver(name, args):
        return err(f"'{name}' denied")
    return registry.call(name, args)

print(production_call(reg, "estimate_refund", {"order_id": "A-3", "pct": "10"}, auto_approver))
print(production_call(reg, "estimate_refund", {"order_id": "A-3", "pct": 200}, auto_approver))

## 🧠 Key takeaways

1. A tool = **name + description + JSON-Schema + function**; the *description* is what makes it discoverable.
2. **Validate arguments before executing** — missing fields, wrong types, enums, ranges.
3. Always return a **consistent envelope** (`{ok, result|error}`); never leak a raw traceback to the model.
4. A **ToolRegistry** centralises dispatch and logs every call for debugging and cost control.
5. Gate **sensitive** tools behind human (or policy) **approval**.
6. Run independent calls **in parallel**, **bound** output size, and make state-changing tools **idempotent**.
7. This exact tool shape is what MCP standardises — Notebook 41.

## ✅ Self-assessment

- [ ] Write a JSON-Schema for a tool's arguments
- [ ] Validate args and produce readable error messages
- [ ] Wrap a tool so it returns a consistent ok/error envelope
- [ ] Build a registry that dispatches by name and logs calls
- [ ] Add an approval gate for a sensitive tool
- [ ] Run several tools in parallel and bound their output

## 🚀 Next step

Continue with **Notebook 41 — The Model Context Protocol (MCP)**, which standardises *exactly* these tools (plus resources and prompts) so any host — Claude Desktop, Claude Code, your own app — can use them over a common wire protocol.